# ANIMA-MECHANICUS — Fregate U-GAMMA : La Forge
> *L'Ame du Mouvement, Forgee par la Machine*

**Pipeline** : `.npz` → `[Blender 4.x headless]` → `[Rig R15 genere + Bake]` → `.fbx (Roblox Studio)`

**Instructions** :
1. Cellule 1 — Installer Blender headless + cloner le codebase
2. Cellule 2 — Uploader les fichiers `.npz` produits par la Fregate U-ALPHA
3. Cellule 3 — Lancer la Forge (Blender headless → .fbx)
4. Cellule 4 — Telecharger les `.fbx` prets pour Roblox Studio

**Note** : Aucun fichier `.blend` template requis — le rig R15 est genere automatiquement.

**Prerequis** : Fichiers `.npz` generes par `ANIMA_MECHANICUS_ALPHA.ipynb`

---

In [ ]:
# ══════ CELLULE 1 — INSTALLATION BLENDER HEADLESS ══════

import os, subprocess

BLENDER_VERSION = "4.2.0"
BLENDER_DIR = f"/content/blender-{BLENDER_VERSION}-linux-x64"
BLENDER_BIN = f"{BLENDER_DIR}/blender"
BLENDER_URL = f"https://download.blender.org/release/Blender4.2/blender-{BLENDER_VERSION}-linux-x64.tar.xz"

# Seule dependance Python de U-GAMMA (etancheite avec U-ALPHA)
os.system("pip install -q numpy")
print("  OK  numpy")

# Telecharger Blender si absent
if not os.path.exists(BLENDER_BIN):
    print(f"[1/2] Telechargement Blender {BLENDER_VERSION} (~180 MB)...")
    result = subprocess.run(
        ["wget", "-q", "--show-progress", "-O", "/tmp/blender.tar.xz", BLENDER_URL]
    )
    if result.returncode != 0:
        raise RuntimeError("Echec telechargement Blender")
    print("[2/2] Extraction...")
    subprocess.run(["tar", "-xf", "/tmp/blender.tar.xz", "-C", "/content/"], check=True)
    os.remove("/tmp/blender.tar.xz")
    print(f"  OK  Blender {BLENDER_VERSION} installe")
else:
    print(f"  OK  Blender {BLENDER_VERSION} deja present")

# Cloner ANIMA-MECHANICUS (script motus_forge.py)
REPO_DIR = "/content/ANIMA-MECHANICUS"
if os.path.exists(REPO_DIR):
    os.system(f"cd {REPO_DIR} && git pull -q")
    print("  OK  Repo ANIMA-MECHANICUS mis a jour")
else:
    os.system(f"git clone -q https://github.com/kioka8877-ux/ANIMA-MECHANICUS.git {REPO_DIR}")
    print("  OK  Repo ANIMA-MECHANICUS clone")

FORGE_SCRIPT = f"{REPO_DIR}/U-GAMMA/codebase/motus_forge.py"
if not os.path.exists(FORGE_SCRIPT):
    raise FileNotFoundError(f"Script forge introuvable : {FORGE_SCRIPT}")

OUTPUT_DIR = "/content/gamma_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\n[U-GAMMA] Installation terminee.")
print(f"  Blender : {BLENDER_BIN}")
print(f"  Script  : {FORGE_SCRIPT}")
print(f"  Outputs : {OUTPUT_DIR}")
print("  Rig R15 : genere automatiquement (aucun .blend template requis)")

In [ ]:
# ══════ CELLULE 2 — UPLOAD DES FICHIERS .npz ══════
# Uploader les fichiers motus_core_P*.npz produits par la Fregate U-ALPHA

from google.colab import files as colab_files
import os

NPZ_INPUT_DIR = "/content/gamma_inputs"
os.makedirs(NPZ_INPUT_DIR, exist_ok=True)

print("Upload des fichiers .npz (produits par ANIMA_MECHANICUS_ALPHA.ipynb) :")
uploaded = colab_files.upload()

NPZ_FILES = []
for fname, data_bytes in uploaded.items():
    if fname.endswith(".npz"):
        dest = f"{NPZ_INPUT_DIR}/{fname}"
        with open(dest, "wb") as f:
            f.write(data_bytes)
        size_kb = os.path.getsize(dest) / 1024
        print(f"  OK  {fname} ({size_kb:.1f} KB)")
        NPZ_FILES.append(dest)

if not NPZ_FILES:
    import glob
    NPZ_GLOB = ""  # @param {type:"string"}
    if NPZ_GLOB:
        NPZ_FILES = sorted(glob.glob(NPZ_GLOB))

if not NPZ_FILES:
    print("\n[ERREUR] Aucun fichier .npz charge.")
    print("  Uploader les fichiers motus_core_P*.npz de la Fregate U-ALPHA")
else:
    import numpy as np
    print(f"\n[U-GAMMA] {len(NPZ_FILES)} fichier(s) .npz prets :")
    for f in NPZ_FILES:
        d = np.load(f, allow_pickle=True)
        n_frames = d["rotations"].shape[0]
        fps_v = int(d["fps"])
        dur = float(d["duration"])
        print(f"  {os.path.basename(f)} | {n_frames} frames @ {fps_v} FPS | {dur:.1f}s source")

In [ ]:
# ══════ CELLULE 3 — FORGE FBX (Blender headless) ══════
# Le rig R15 est cree programmatiquement — aucun template .blend requis
# Duree estimee : 30-90 secondes par fichier .npz

import os, subprocess

assert NPZ_FILES, "[ERREUR] Charger les .npz en Cellule 2"

FBX_FILES = []

for npz_path in NPZ_FILES:
    basename = os.path.basename(npz_path).replace(".npz", ".fbx").replace("motus_core_", "ANIMA_MECHANICUS_")
    fbx_out = f"{OUTPUT_DIR}/{basename}"

    print(f"\n[FORGE] {os.path.basename(npz_path)} → {basename}")

    # Appel Blender headless — rig R15 genere automatiquement dans motus_forge.py
    result = subprocess.run(
        [
            BLENDER_BIN, "--background",
            "--python", FORGE_SCRIPT,
            "--", npz_path, fbx_out
        ],
        capture_output=True, text=True
    )

    # Afficher les lignes [U-GAMMA] du stdout Blender
    for line in result.stdout.splitlines():
        if "[U-GAMMA]" in line or "[ERREUR]" in line:
            print(f"  {line}")

    if result.returncode != 0 or not os.path.exists(fbx_out):
        print(f"  [ERREUR] Blender a echoue pour {os.path.basename(npz_path)}")
        for line in result.stderr.strip().splitlines()[-15:]:
            print(f"  STDERR: {line}")
        continue

    size_kb = os.path.getsize(fbx_out) / 1024
    print(f"  OK  {basename} ({size_kb:.1f} KB)")
    FBX_FILES.append(fbx_out)

print(f"\n[U-GAMMA] {len(FBX_FILES)}/{len(NPZ_FILES)} fichier(s) .fbx forges")
if FBX_FILES:
    print("Passer a la Cellule 4 pour telecharger les .fbx")

In [ ]:
# ══════ CELLULE 4 — TELECHARGEMENT DES .fbx ══════

from google.colab import files as colab_files
import os

assert FBX_FILES, "[ERREUR] Aucun .fbx genere — relancer la Cellule 3"

print(f"[U-GAMMA] {len(FBX_FILES)} fichier(s) .fbx prets :")
for fbx_path in FBX_FILES:
    size_kb = os.path.getsize(fbx_path) / 1024
    print(f"  {os.path.basename(fbx_path)} ({size_kb:.1f} KB)")

print("\nTelechargement en cours...")
for fbx_path in FBX_FILES:
    colab_files.download(fbx_path)

print("\n[U-GAMMA] Forge terminee — Mission accomplie.")
print("")
print("Import dans Roblox Studio :")
print("  1. Ouvrir Roblox Studio")
print("  2. Plugins → Avatar Importer (ou Import 3D)")
print("  3. Selectionner le .fbx telecharge")
print("  4. L'animation R15 est prete a l'emploi")